# Neuro-Symbolic CSFM

In [ ]:
# 1. Install PyTorch Geometric and NetworkX
!pip install torch_geometric networkx
!pip install torch-scatter -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+$(python -c "import torch; print('cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() and torch.version.cuda else 'cpu')").html

# 2. Download and compile Berkeley ABC (Logic Synthesis Engine)
!git clone https://github.com/berkeley-abc/abc.git
%cd abc
!make ABC_USE_NO_READLINE=1 -j4
%cd /content

# 3. Download the EPFL Combinational Benchmarks
!git clone https://github.com/lsils/benchmarks.git epfl_benchmarks
## RESTART

In [ ]:
#@title ns csfm
%%writefile csfm_eda_framework.py
import os
import sys
import time
import glob
import random
import hashlib
import argparse
import subprocess
from collections import deque
from dataclasses import dataclass
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GraphSAGE, global_mean_pool, global_max_pool

# =====================================================================
# PATHS & CONFIGURATION
# =====================================================================
ABC_EXE = os.environ.get("ABC_BINARY", "/content/abc/abc")
EPFL_DIR = os.environ.get("EPFL_DIR", "/content/epfl_benchmarks")

if not os.path.exists(ABC_EXE):
    raise FileNotFoundError(f"ABC binary not found at '{ABC_EXE}'. Please build ABC first.")

# Standard industrial heuristic scripts
RESYN2 = "balance; rewrite; refactor; balance; rewrite; rewrite -z; balance; refactor -z; rewrite -z; balance"
COMPRESS2RS = "b -l; rw -l; rf -l; b -l; rw -l; rw -z -l; b -l; rf -z -l; rw -z -l; b -l"
MIXED_OPT = "balance; rewrite -l; refactor; balance -l"

# =====================================================================
# METRICS LOGGER
# =====================================================================
@dataclass
class CircuitMetrics:
    name: str
    init_area: int; init_delay: int; init_adp: int
    opt_area: int; opt_delay: int; opt_adp: int
    delta_area_pct: float; delta_delay_pct: float; delta_adp_pct: float
    neural_time_sec: float; symbolic_time_sec: float; total_time_sec: float
    cec_passed: bool

class JournalMetricsLogger:
    def __init__(self):
        self.results: List[CircuitMetrics] = []

    def record(self, metric: CircuitMetrics):
        self.results.append(metric)

    def print_terminal_table(self):
        print("\n" + "=" * 125)
        print(f"{'Benchmark':<14} | {'Initial (A / D)':<16} | {'Optimized (A / D)':<17} | {'Δ Area (%)':<11} | {'Δ Delay (%)':<12} | {'Δ ADP (%)':<10} | {'T_tot (s)':<9} | {'CEC'}")
        print("-" * 125)
        for r in self.results:
            init_str = f"{r.init_area} / {r.init_delay}"
            opt_str = f"{r.opt_area} / {r.opt_delay}"
            cec_str = "PASS" if r.cec_passed else "FAIL"
            print(f"{r.name:<14} | {init_str:<16} | {opt_str:<17} | {r.delta_area_pct:>10.2f}% | {r.delta_delay_pct:>11.2f}% | {r.delta_adp_pct:>9.2f}% | {r.total_time_sec:>8.2f}s | {cec_str}")
        print("=" * 125)

# =====================================================================
# INDUSTRIAL AIG PARSER (O(V+E) TOPOLOGICAL EXTRACTION & EXACT HASHING)
# =====================================================================
class RealAIGState:
    def __init__(self, filepath: str):
        self.filepath = os.path.abspath(filepath)
        self._node_count: Optional[int] = None
        self._level_count: Optional[int] = None
        self._pyg_data: Optional[Data] = None

    def _fetch_stats(self):
        result = subprocess.run(f'{ABC_EXE} -c "read {self.filepath}; print_stats"', shell=True, capture_output=True, text=True)
        try:
            self._node_count = int(result.stdout.split("and =")[1].split()[0])
            self._level_count = int(result.stdout.split("lev =")[1].split()[0])
        except (IndexError, ValueError):
            self._node_count, self._level_count = 999999, 999999

    @property
    def node_count(self) -> int:
        if self._node_count is None: self._fetch_stats()
        return self._node_count

    @property
    def level_count(self) -> int:
        if self._level_count is None: self._fetch_stats()
        return self._level_count

    @property
    def adp(self) -> int:
        return self.node_count * self.level_count

    def structural_hash(self) -> str:
        """ Cryptographically secure state deduplication avoiding scalar collisions. """
        tmp_aig = self.filepath + f"_hash_{random.randint(0,99999)}.aiger"
        subprocess.run(f'{ABC_EXE} -c "read {self.filepath}; write_aiger {tmp_aig}"', shell=True, capture_output=True)
        if os.path.exists(tmp_aig):
            with open(tmp_aig, "rb") as f:
                shash = hashlib.sha256(f.read()).hexdigest()
            os.remove(tmp_aig)
            return shash
        return str(time.time())

    def to_pyg_data(self) -> Data:
        if self._pyg_data is not None:
            return self._pyg_data

        tmp_bench = self.filepath + ".bench"
        subprocess.run(f'{ABC_EXE} -c "read {self.filepath}; strash; write_bench {tmp_bench}"', shell=True, capture_output=True)

        edges, node_types, node_map = [], {}, {}
        is_po = set()

        def get_id(name, default_type):
            if name not in node_map:
                node_map[name] = len(node_map)
                node_types[node_map[name]] = default_type
            return node_map[name]

        with open(tmp_bench, 'r') as f: lines = f.readlines()

        for line in lines:
            line = line.strip()
            if line.startswith("INPUT"): get_id(line.split("(")[1].split(")")[0].strip(), 0)

        for line in lines:
            line = line.strip()
            if "=" in line:
                out_node, logic = [part.strip() for part in line.split("=")]
                n_type = 2 if "NOT" in logic else 1
                out_id = get_id(out_node, n_type)
                node_types[out_id] = n_type
                if "(" in logic:
                    for inp in logic.split("(")[1].split(")")[0].split(","):
                        edges.append((get_id(inp.strip(), 0), out_id))

        for line in lines:
            line = line.strip()
            if line.startswith("OUTPUT"):
                out_name = line.split("(")[1].split(")")[0].strip()
                if out_name in node_map:
                    is_po.add(node_map[out_name])

        if os.path.exists(tmp_bench): os.remove(tmp_bench)
        if not edges: return Data(x=torch.zeros((1, 5)), edge_index=torch.empty((2, 0), dtype=torch.long))

        # O(V + E) Kahn's Algorithm Implementation via deque
        adj = {i: [] for i in range(len(node_map))}
        in_degree = {i: 0 for i in range(len(node_map))}
        for u, v in edges:
            adj[u].append(v)
            in_degree[v] += 1

        levels = {i: 0 for i in range(len(node_map))}
        queue = deque([i for i in range(len(node_map)) if in_degree[i] == 0])

        while queue:
            curr = queue.popleft()
            for v in adj[curr]:
                # True AIG Depth: Inverters (Type 2) do not increase logic delay
                inc = 1 if node_types.get(v, 1) == 1 else 0
                levels[v] = max(levels[v], levels[curr] + inc)
                in_degree[v] -= 1
                if in_degree[v] == 0: queue.append(v)

        max_level = max(levels.values()) if levels else 1
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

        # Node features: [PI, AND, NOT, PO, Normalized_Level]
        x = torch.zeros((len(node_map), 5), dtype=torch.float)
        for nid, n_type in node_types.items():
            x[nid, n_type] = 1.0
            if nid in is_po: x[nid, 3] = 1.0
            x[nid, 4] = levels[nid] / max_level

        self._pyg_data = Data(x=x, edge_index=edge_index)
        return self._pyg_data

    def get_legal_moves(self, workspace_dir: str, step_id: str, action_set: List[str]) -> List['RealAIGState']:
        moves = []
        for i, action in enumerate(action_set):
            out_file = os.path.join(workspace_dir, f"state_{step_id}_{i}.aig")
            subprocess.run(f'{ABC_EXE} -c "read {self.filepath}; strash; {action}; write {out_file}"', shell=True, capture_output=True)
            if os.path.exists(out_file): moves.append(RealAIGState(out_file))
        return moves

    def cec_verify(self, target_state: 'RealAIGState') -> bool:
        result = subprocess.run(f'{ABC_EXE} -c "cec {self.filepath} {target_state.filepath}"', shell=True, capture_output=True, text=True)
        return "Networks are equivalent" in result.stdout

# =====================================================================
# NEURAL ARCHITECTURES (\psi: Encoder, \phi: Source, \theta: Flow)
# =====================================================================
class DualPoolAIGEncoder(nn.Module):
    def __init__(self, node_features=5, hidden_dim=64, latent_dim=128):
        super().__init__()
        self.gnn = GraphSAGE(in_channels=node_features, hidden_channels=hidden_dim, num_layers=3, out_channels=hidden_dim)
        self.proj = nn.Linear(hidden_dim * 2, latent_dim)

    def forward(self, batch_data):
        h = self.gnn(batch_data.x, batch_data.edge_index)
        h_mean = global_mean_pool(h, batch_data.batch)
        h_max = global_max_pool(h, batch_data.batch)
        return self.proj(torch.cat([h_mean, h_max], dim=-1))

class ParetoCSFMSourceGenerator(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(latent_dim + 2, 256), nn.GELU(), nn.Linear(256, 256), nn.GELU())
        self.mu_head = nn.Linear(256, latent_dim)
        self.logvar_head = nn.Linear(256, latent_dim)

    def forward(self, c, w):
        h = self.net(torch.cat([c, w], dim=-1))
        return self.mu_head(h), self.logvar_head(h)

class ParetoCSFMVelocityModel(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(latent_dim * 2 + 3, 512), nn.GELU(), nn.Linear(512, 512), nn.GELU(), nn.Linear(512, latent_dim))

    def forward(self, xt, c, w, t):
        t_vec = t.expand(xt.size(0), 1) if t.dim() == 1 else t
        return self.net(torch.cat([xt, c, w, t_vec], dim=-1))

# =====================================================================
# NEURO-SYMBOLIC ENGINE
# =====================================================================
class NeuroSymbolicEDAEngine:
    def __init__(self, args):
        self.args = args
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.workspace = os.path.abspath(args.workspace)
        os.makedirs(self.workspace, exist_ok=True)

        self.encoder = DualPoolAIGEncoder(latent_dim=args.latent_dim).to(self.device)
        self.source_gen = ParetoCSFMSourceGenerator(latent_dim=args.latent_dim).to(self.device)
        self.flow_model = ParetoCSFMVelocityModel(latent_dim=args.latent_dim).to(self.device)

        self.optimizer = optim.AdamW(
            list(self.encoder.parameters()) + list(self.source_gen.parameters()) + list(self.flow_model.parameters()),
            lr=args.lr, weight_decay=1e-4
        )
        self.action_library = ["rewrite", "refactor", "balance", "rewrite -z", "refactor -z", "balance -l", "rewrite -l"]

    def _get_pareto_target(self, pool: List[RealAIGState], w: torch.Tensor, base_a: int, base_d: int) -> RealAIGState:
        """ Continuously selects the optimal target based on sampled Dirichlet preference weights. """
        def score(s): return w[0].item() * (s.node_count / base_a) + w[1].item() * (s.level_count / base_d)
        return min(pool, key=score)

    def prepare_training_pairs(self, circuit_files: List[str]):
        print(f"\n[+] Extracting Training Data (Sampling {self.args.num_pareto_samples} continuous Pareto points per circuit)...")
        dataset = []
        for idx, file in enumerate(circuit_files):
            base = os.path.basename(file).split(".")[0]
            unopt, opt_a, opt_d, opt_m = [os.path.join(self.workspace, f"{base}_{s}.aig") for s in ["unopt", "opt_a", "opt_d", "opt_m"]]

            subprocess.run(f'{ABC_EXE} -c "read {file}; strash; write {unopt}"', shell=True, capture_output=True)
            subprocess.run(f'{ABC_EXE} -c "read {file}; strash; {RESYN2}; write {opt_a}"', shell=True, capture_output=True)
            subprocess.run(f'{ABC_EXE} -c "read {file}; strash; {COMPRESS2RS}; write {opt_d}"', shell=True, capture_output=True)
            subprocess.run(f'{ABC_EXE} -c "read {file}; strash; {MIXED_OPT}; write {opt_m}"', shell=True, capture_output=True)

            if all(os.path.exists(f) for f in [unopt, opt_a, opt_d, opt_m]):
                su, sa, sd, sm = RealAIGState(unopt), RealAIGState(opt_a), RealAIGState(opt_d), RealAIGState(opt_m)
                if su.node_count < 999999 and sa.node_count < 999999:
                    su.to_pyg_data(); sa.to_pyg_data(); sd.to_pyg_data(); sm.to_pyg_data()
                    base_a, base_d = max(su.node_count, 1), max(su.level_count, 1)
                    target_pool = [su, sa, sd, sm]

                    for _ in range(self.args.num_pareto_samples):
                        w = torch.rand(2); w /= w.sum() # Dirichlet sample
                        target_state = self._get_pareto_target(target_pool, w, base_a, base_d)
                        dataset.append((su, target_state, w))

                    print(f"  [{idx+1}/{len(circuit_files)}] Loaded {base} | Base: {su.node_count} nodes")
        return dataset

    def train_csfm(self, dataset, epochs: int):
        print(f"\n[+] Training CSFM Model (Batch Size: {self.args.batch_size})")
        self.encoder.train(); self.source_gen.train(); self.flow_model.train()

        for epoch in range(1, epochs + 1):
            random.shuffle(dataset)
            ep_loss = 0.0

            for i in range(0, len(dataset), self.args.batch_size):
                batch_items = dataset[i:i + self.args.batch_size]

                c_batch = Batch.from_data_list([item[0]._pyg_data for item in batch_items]).to(self.device)
                x1_batch = Batch.from_data_list([item[1]._pyg_data for item in batch_items]).to(self.device)
                w_tensor = torch.stack([item[2] for item in batch_items]).to(self.device)

                self.optimizer.zero_grad()

                # Encoder (\psi)
                C, X1 = self.encoder(c_batch), self.encoder(x1_batch)

                # Source Generator (\phi)
                mu, logvar = self.source_gen(C, w_tensor)
                sigma = torch.exp(0.5 * logvar)
                X0 = mu + sigma * torch.randn_like(mu)

                # Flow Integration (\theta)
                t = torch.rand(X0.size(0), 1, device=self.device)
                Xt = (1.0 - t) * X0 + t * X1
                v_pred = self.flow_model(Xt, C, w_tensor, t)
                v_target = X1 - X0

                # Objectives
                L_FM = torch.mean(torch.sum((v_pred - v_target) ** 2, dim=-1))
                L_VarReg = 0.5 * torch.mean(sigma.pow(2) - 1.0 - logvar)
                L_align = torch.mean(1.0 - F.cosine_similarity(X0, X1, dim=-1))

                # VICReg-style latent standard deviation penalty (\psi)
                std_C = torch.sqrt(C.var(dim=0) + 1e-04)
                L_std = torch.mean(F.relu(1.0 - std_C))

                loss = L_FM + (self.args.l_varreg * L_VarReg) + (self.args.l_align * L_align) + (self.args.l_std * L_std)
                loss.backward()
                self.optimizer.step()
                ep_loss += loss.item()

            if epoch % 5 == 0 or epoch == 1:
                print(f"  Epoch {epoch:02d}/{epochs:02d} | Avg Loss: {ep_loss / (len(dataset)//self.args.batch_size + 1):.4f}")

    @torch.no_grad()
    def heuristic_beam_search(self, initial_state: RealAIGState, w: torch.Tensor) -> Tuple[RealAIGState, float, float]:
        t_start = time.perf_counter()
        self.encoder.eval(); self.source_gen.eval(); self.flow_model.eval()

        # 1. Deterministic Neural Rollout
        C = self.encoder(Batch.from_data_list([initial_state.to_pyg_data()]).to(self.device))
        w_tensor = w.unsqueeze(0).to(self.device)
        mu, _ = self.source_gen(C, w_tensor)
        target_z, dt = mu, 1.0 / self.args.ode_steps
        for step in range(self.args.ode_steps):
            target_z = target_z + self.flow_model(target_z, C, w_tensor, torch.tensor([[step * dt]], device=self.device)) * dt
        t_neural = time.perf_counter() - t_start

        # 2. Heuristic Beam Search
        t_search_start = time.perf_counter()
        alpha_area, alpha_delay = w[0].item(), w[1].item()
        base_nodes, base_levels = max(initial_state.node_count, 1), max(initial_state.level_count, 1)

        def compute_g(s: RealAIGState) -> float:
            return alpha_area * (s.node_count / base_nodes) + alpha_delay * (s.level_count / base_levels)

        beam = [(compute_g(initial_state), compute_g(initial_state), initial_state)]
        best_state, best_g = initial_state, compute_g(initial_state)
        visited_hashes = set([initial_state.structural_hash()])
        created_temp_files = []

        print(f"\n  Starting Beam Search | Depth {self.args.search_depth} | Width {self.args.beam_width} | Heuristic W: {self.args.heuristic_weight}")

        for depth in range(self.args.search_depth):
            candidates = []
            for _, _, curr_state in beam:
                moves = curr_state.get_legal_moves(self.workspace, f"d{depth}_{random.randint(0, 9999)}", self.action_library)
                created_temp_files.extend([m.filepath for m in moves])

                for nxt in moves:
                    shash = nxt.structural_hash()
                    if shash in visited_hashes: continue
                    visited_hashes.add(shash)

                    g_score = compute_g(nxt)
                    try:
                        z_curr = self.encoder(Batch.from_data_list([nxt.to_pyg_data()]).to(self.device))
                        h_score = self.args.heuristic_weight * (1.0 - F.cosine_similarity(z_curr, target_z, dim=-1).item())
                    except Exception:
                        h_score = 9999.0

                    candidates.append((g_score + h_score, g_score, nxt))
                    if g_score < best_g: best_g, best_state = g_score, nxt

            candidates.sort(key=lambda x: x[0])
            beam = candidates[:self.args.beam_width]
            if not beam: break
            print(f"    Step {depth+1:02d} | Beam Top Cost: {beam[0][0]:.4f} | Best Area: {best_state.node_count} | Best Delay: {best_state.level_count}")

        for f in set(created_temp_files):
            if f != best_state.filepath and f != initial_state.filepath and os.path.exists(f):
                try: os.remove(f)
                except OSError: pass

        return best_state, t_neural, time.perf_counter() - t_search_start

# =====================================================================
# CLI LEVERS & USER CONFIG
# =====================================================================
def get_user_levers():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epfl_dir", type=str, default=EPFL_DIR)
    parser.add_argument("--workspace", type=str, default="/content/workspace")
    parser.add_argument("--train_subset", type=int, default=15)
    parser.add_argument("--test_subset", type=int, default=5)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--num_pareto_samples", type=int, default=4)
    parser.add_argument("--alpha_area", type=float, default=0.5)
    parser.add_argument("--alpha_delay", type=float, default=0.5)
    parser.add_argument("--latent_dim", type=int, default=128)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--lr", type=float, default=2e-3)
    parser.add_argument("--l_varreg", type=float, default=5.0)
    parser.add_argument("--l_align", type=float, default=1.0)
    parser.add_argument("--l_std", type=float, default=0.5) 
    parser.add_argument("--ode_steps", type=int, default=50)
    parser.add_argument("--search_depth", type=int, default=10)
    parser.add_argument("--beam_width", type=int, default=4)
    parser.add_argument("--heuristic_weight", type=float, default=2.0)
    args, _ = parser.parse_known_args()
    return args

def main():
    args = get_user_levers()
    logger = JournalMetricsLogger()
    total_w = args.alpha_area + args.alpha_delay
    w_vec = torch.tensor([args.alpha_area / total_w, args.alpha_delay / total_w], dtype=torch.float)

    print("=" * 80)
    print("🔬 SCALABLE NEURO-SYMBOLIC CSFM SYNTHESIS")
    print(f"Objective: {w_vec[0].item():.2f} Area / {w_vec[1].item():.2f} Delay | Batch Size: {args.batch_size}")
    print("=" * 80)

    engine = NeuroSymbolicEDAEngine(args)
    all_circuits = glob.glob(os.path.join(args.epfl_dir, "**", "*.aig"), recursive=True)
    random.seed(42)
    random.shuffle(all_circuits)

    if len(all_circuits) < args.train_subset + args.test_subset:
        raise ValueError("Not enough EPFL .aig circuits found. Clone the benchmark suite.")

    train_circuits = all_circuits[:args.train_subset]
    test_circuits = all_circuits[args.train_subset : args.train_subset + args.test_subset]

    training_dataset = engine.prepare_training_pairs(train_circuits)
    engine.train_csfm(training_dataset, epochs=args.epochs)

    print("\n" + "=" * 80)
    print(f"🚀 EVALUATING {len(test_circuits)} UNSEEN BENCHMARKS")
    print("=" * 80)

    for test_file in test_circuits:
        name = os.path.basename(test_file).replace(".aig", "")
        test_canon = os.path.join(engine.workspace, f"{name}_eval_init.aig")
        subprocess.run(f'{ABC_EXE} -c "read {test_file}; strash; write {test_canon}"', shell=True, capture_output=True)

        initial_state = RealAIGState(test_canon)
        print(f"\n[Test] Circuit: {name.upper()} | Initial (Area: {initial_state.node_count}, Delay: {initial_state.level_count})")

        optimized_state, t_neural, t_symbolic = engine.heuristic_beam_search(initial_state, w_vec)

        metric = CircuitMetrics(
            name=name, init_area=initial_state.node_count, init_delay=initial_state.level_count, init_adp=initial_state.adp,
            opt_area=optimized_state.node_count, opt_delay=optimized_state.level_count, opt_adp=optimized_state.adp,
            delta_area_pct=100.0 * (1.0 - (optimized_state.node_count / max(initial_state.node_count, 1))),
            delta_delay_pct=100.0 * (1.0 - (optimized_state.level_count / max(initial_state.level_count, 1))),
            delta_adp_pct=100.0 * (1.0 - (optimized_state.adp / max(initial_state.adp, 1))),
            neural_time_sec=t_neural, symbolic_time_sec=t_symbolic, total_time_sec=t_neural+t_symbolic,
            cec_passed=initial_state.cec_verify(optimized_state)
        )
        logger.record(metric)

    logger.print_terminal_table()

if __name__ == "__main__":
    main()

Writing csfm_eda_framework.py


In [12]:
!python csfm_eda_framework.py \
  --alpha_area 0.5 \
  --alpha_delay 0.5 \
  --train_subset 15 \
  --test_subset 5 \
  --batch_size 32 \
  --num_pareto_samples 4 \
  --epochs 50 \
  --search_depth 10 \
  --beam_width 4

🔬 SCALABLE NEURO-SYMBOLIC CSFM SYNTHESIS
Objective: 0.50 Area / 0.50 Delay | Batch Size: 32

[+] Extracting Training Data (Sampling 4 continuous Pareto points per circuit)...
  [1/15] Loaded multiplier | Base: 27062 nodes
  [2/15] Loaded mem_ctrl | Base: 46836 nodes
  [3/15] Loaded adder | Base: 1020 nodes
  [4/15] Loaded cavlc | Base: 693 nodes
  [5/15] Loaded voter | Base: 13758 nodes
  [6/15] Loaded sqrt | Base: 24618 nodes
  [7/15] Loaded max | Base: 2865 nodes
  [8/15] Loaded square | Base: 18484 nodes
  [9/15] Loaded i2c | Base: 1342 nodes
  [10/15] Loaded hyp | Base: 214335 nodes
  [11/15] Loaded bar | Base: 3336 nodes
  [12/15] Loaded log2 | Base: 32060 nodes
  [13/15] Loaded router | Base: 257 nodes
  [14/15] Loaded sin | Base: 5416 nodes
  [15/15] Loaded int2float | Base: 260 nodes

[+] Training CSFM Model (Batch Size: 32)
  Epoch 01/50 | Avg Loss: 128.2874
  Epoch 05/50 | Avg Loss: 40.9432
  Epoch 10/50 | Avg Loss: 12.1331
  Epoch 15/50 | Avg Loss: 10.9423
  Epoch 20/50 | Av

In [13]:
!python csfm_eda_framework.py \
  --alpha_area 0.5 \
  --alpha_delay 0.5 \
  --train_subset 15 \
  --test_subset 5 \
  --batch_size 32 \
  --heuristic_weight 0.0 \
  --search_depth 10 \
  --beam_width 4

🔬 SCALABLE NEURO-SYMBOLIC CSFM SYNTHESIS
Objective: 0.50 Area / 0.50 Delay | Batch Size: 32

[+] Extracting Training Data (Sampling 4 continuous Pareto points per circuit)...
  [1/15] Loaded multiplier | Base: 27062 nodes
  [2/15] Loaded mem_ctrl | Base: 46836 nodes
  [3/15] Loaded adder | Base: 1020 nodes
  [4/15] Loaded cavlc | Base: 693 nodes
  [5/15] Loaded voter | Base: 13758 nodes
  [6/15] Loaded sqrt | Base: 24618 nodes
  [7/15] Loaded max | Base: 2865 nodes
  [8/15] Loaded square | Base: 18484 nodes
  [9/15] Loaded i2c | Base: 1342 nodes
  [10/15] Loaded hyp | Base: 214335 nodes
  [11/15] Loaded bar | Base: 3336 nodes
  [12/15] Loaded log2 | Base: 32060 nodes
  [13/15] Loaded router | Base: 257 nodes
  [14/15] Loaded sin | Base: 5416 nodes
  [15/15] Loaded int2float | Base: 260 nodes

[+] Training CSFM Model (Batch Size: 32)
  Epoch 01/50 | Avg Loss: 131.0795
  Epoch 05/50 | Avg Loss: 51.2203
  Epoch 10/50 | Avg Loss: 12.3779
  Epoch 15/50 | Avg Loss: 11.1869
  Epoch 20/50 | Av

In [14]:
# Ablation A: No Directional Alignment (Expect erratic trajectory)
!python csfm_eda_framework.py \
  --alpha_area 0.5 \
  --alpha_delay 0.5 \
  --train_subset 15 \
  --test_subset 3 \
  --batch_size 32 \
  --l_align 0.0 \
  --l_varreg 5.0

# Ablation B: No Variance Regularization (Expect mode collapse)
!python csfm_eda_framework.py \
  --alpha_area 0.5 \
  --alpha_delay 0.5 \
  --train_subset 15 \
  --test_subset 3 \
  --batch_size 32 \
  --l_align 1.0 \
  --l_varreg 0.0

🔬 SCALABLE NEURO-SYMBOLIC CSFM SYNTHESIS
Objective: 0.50 Area / 0.50 Delay | Batch Size: 32

[+] Extracting Training Data (Sampling 4 continuous Pareto points per circuit)...
  [1/15] Loaded multiplier | Base: 27062 nodes
  [2/15] Loaded mem_ctrl | Base: 46836 nodes
  [3/15] Loaded adder | Base: 1020 nodes
  [4/15] Loaded cavlc | Base: 693 nodes
  [5/15] Loaded voter | Base: 13758 nodes
  [6/15] Loaded sqrt | Base: 24618 nodes
  [7/15] Loaded max | Base: 2865 nodes
  [8/15] Loaded square | Base: 18484 nodes
  [9/15] Loaded i2c | Base: 1342 nodes
  [10/15] Loaded hyp | Base: 214335 nodes
  [11/15] Loaded bar | Base: 3336 nodes
  [12/15] Loaded log2 | Base: 32060 nodes
  [13/15] Loaded router | Base: 257 nodes
  [14/15] Loaded sin | Base: 5416 nodes
  [15/15] Loaded int2float | Base: 260 nodes

[+] Training CSFM Model (Batch Size: 32)
  Epoch 01/50 | Avg Loss: 127.3911
  Epoch 05/50 | Avg Loss: 41.1980
  Epoch 10/50 | Avg Loss: 12.0736
  Epoch 15/50 | Avg Loss: 10.7493
  Epoch 20/50 | Av

In [15]:
# Area-Heavy Sweep
!python csfm_eda_framework.py \
  --alpha_area 0.9 \
  --alpha_delay 0.1 \
  --train_subset 15 \
  --test_subset 5 \
  --batch_size 32

# Delay-Heavy Sweep
!python csfm_eda_framework.py \
  --alpha_area 0.1 \
  --alpha_delay 0.9 \
  --train_subset 15 \
  --test_subset 5 \
  --batch_size 32

🔬 SCALABLE NEURO-SYMBOLIC CSFM SYNTHESIS
Objective: 0.90 Area / 0.10 Delay | Batch Size: 32

[+] Extracting Training Data (Sampling 4 continuous Pareto points per circuit)...
  [1/15] Loaded multiplier | Base: 27062 nodes
  [2/15] Loaded mem_ctrl | Base: 46836 nodes
  [3/15] Loaded adder | Base: 1020 nodes
  [4/15] Loaded cavlc | Base: 693 nodes
  [5/15] Loaded voter | Base: 13758 nodes
  [6/15] Loaded sqrt | Base: 24618 nodes
  [7/15] Loaded max | Base: 2865 nodes
  [8/15] Loaded square | Base: 18484 nodes
  [9/15] Loaded i2c | Base: 1342 nodes
  [10/15] Loaded hyp | Base: 214335 nodes
  [11/15] Loaded bar | Base: 3336 nodes
  [12/15] Loaded log2 | Base: 32060 nodes
  [13/15] Loaded router | Base: 257 nodes
  [14/15] Loaded sin | Base: 5416 nodes
  [15/15] Loaded int2float | Base: 260 nodes

[+] Training CSFM Model (Batch Size: 32)
  Epoch 01/50 | Avg Loss: 127.3628
  Epoch 05/50 | Avg Loss: 43.9398
  Epoch 10/50 | Avg Loss: 12.2220
  Epoch 15/50 | Avg Loss: 11.3681
  Epoch 20/50 | Av